# **Chroma DB**

This is a exploratory notebook to understand the functionalities of **Chroma DB**. 

In [ ]:
#%pip install langchain langchain-openai langchain-community chromadb

### **1. Docling**
*Convert PDFs into Chunks*

In [4]:
from langchain_docling import DoclingLoader
from docling.chunking import HybridChunker

FILE_PATH = r"../my_papers/2024 - 1.pdf"

loader = DoclingLoader(file_path=FILE_PATH, chunker=HybridChunker())

docs = loader.load()

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-02-16 16:24:56,425 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-16 16:24:56,439 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-16 16:24:56,440 [RapidOCR] main.py:53: Using C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-16 16:24:56,565 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-16 16:24:56,569 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-16 16:24:56,570 [Ra

***Organise chunked information properly for embeddings and storage in Chroma DB***

In [5]:
from langchain_core.documents import Document
import os
import uuid


def convert_docling_chunks(doc_splits):

    clean_documents = []

    for i, doc in enumerate(doc_splits):

        dl_meta = doc.metadata.get("dl_meta", {})
        origin = dl_meta.get("origin", {})
        headings = dl_meta.get("headings", [])
        doc_items = dl_meta.get("doc_items", [])

        # Extract page number safely
        page_no = None
        if doc_items:
            prov = doc_items[0].get("prov", [])
            if prov:
                page_no = prov[0].get("page_no")

        filename = origin.get("filename")
        paper_id = os.path.splitext(filename)[0]

        clean_metadata = {
            "paper_id": paper_id,
            "filename": filename,
            "section_heading": headings[0] if headings else None,
            "page_no": page_no,
            "modality": "text",  # later you can detect table/figure
            "chunk_id": f"{paper_id}_{i}"
        }

        clean_doc = Document(
            page_content=doc.page_content,
            metadata=clean_metadata
        )

        clean_documents.append(clean_doc)

    return clean_documents


In [19]:
documents = convert_docling_chunks(docs)

TypeError: expected str, bytes or os.PathLike object, not NoneType

In [9]:
print(documents[0])

page_content='Contents lists available at ScienceDirect' metadata={'paper_id': '2024 - 1', 'filename': '2024 - 1.pdf', 'section_heading': None, 'page_no': 1, 'modality': 'text', 'chunk_id': '2024 - 1_0'}


### **2. Embeddings and Chroma DB Storage**

Now Store the Chunks in Persistent Chroma

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

PERSIST_DIRECTORY = "/vector_database" # Set this to your desired path

def store_documents(documents):

    vectorstore = Chroma(
        collection_name="theory_synthesis", # Change name if changed emebedding model
        embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
        persist_directory=PERSIST_DIRECTORY
    )

    vectorstore.add_documents(documents)
    vectorstore.persist()

    print("✅ Stored successfully.")


In [14]:
data = store_documents(documents)

✅ Stored successfully.


Retrieve only one paper

In [16]:
def get_paper_retriever(paper_id, k=6):

    vectorstore = Chroma(
        collection_name="theory_synthesis",
        embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
        persist_directory="./chroma_db"
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={
            "k": k,
            "filter": {"paper_id": paper_id}
        }
    )

    return retriever


Usage

In [18]:
retriever = get_paper_retriever("2024 - 1")

docs = retriever.invoke(
    "What methodology was used?"
)

for d in docs:
    print(d.metadata)
    print(d.page_content[:300])


{'section_heading': '5.3.1. Questionnaire distribution and analysis', 'page_no': 8, 'paper_id': '2024 - 1', 'chunk_id': '2024 - 1_59', 'modality': 'text', 'filename': '2024 - 1.pdf'}
5.3.1. Questionnaire distribution and analysis
Based  on  the  above  analysis,  a  second-round  questionnaire  was designed using 17 retained items on cultural resilience and predictive variables (i.e., maturity of the digital strategy and cultural sustainability)  for  subsequent  nomological  an
{'chunk_id': '2024 - 1_56', 'paper_id': '2024 - 1', 'filename': '2024 - 1.pdf', 'section_heading': '5.2.1. Questionnaire collection', 'page_no': 8, 'modality': 'text'}
5.2.1. Questionnaire collection
The first round of questionnaire distribution combined a field survey (the ancient city of Quanzhou) and an online survey (Sojump) in May 2023. 388 valid questionnaires were obtained (Nunnally, 1967). Sixty invalid  questionnaires  with  random  or  incomplete  answer
{'page_no': 4, 'chunk_id': '2024 - 1_35', 'file

# Future Addition Note

🔎 **Future Update: Adding Figures Without Reprocessing**

- Vector DB entries are independent → new content can be appended anytime.

- No need to re-chunk or re-embed existing text.

- Add figure descriptions as new documents with:

    - unique element_id
    - content_type = "figure"
    - same embedding model (must remain consistent).

- Keep everything in one collection and use metadata filtering (paper_id, content_type).

⚠️ Only reprocess if you change embedding model or chunking strategy.